# Embedding Consistency Checks

This notebook is for quick embedding experiments with the trained CNN model.

It supports:
- image vs image comparison
- video frame consistency checks
- video vs image comparison

Update the paths in the config cell, then run the cells you need.

In [27]:
from pathlib import Path
import random

import cv2
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from PIL import Image
from tqdm.auto import tqdm
from torchvision import models
from insightface.app import FaceAnalysis
from insightface.utils import face_align

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True

print("device:", device)
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

device: cuda
torch: 2.6.0+cu124
cuda available: True


In [28]:
# -----------------------------
# Config
# -----------------------------
CHECKPOINT_PATH = Path(r"C:\DSP\checkpoints\celeba_embedding_best.pt")
INSIGHTFACE_MODEL_NAME = "buffalo_l"
INSIGHTFACE_ROOT = None
DET_SIZE = (640, 640)
RANDOM_SEED = 42
MAX_VIDEO_FRAMES = 12
FACE_OUT_SIZE = 224
USE_IMAGE_FALLBACK_IF_NO_FACE = True

IMAGE_PATH_A = Path(r"C:\DSP\Crops\20260309_004904_frame8_face0.jpg")
IMAGE_PATH_B = Path(r"C:\DSP\Crops\20260309_004904_frame151_face0.jpg")
VIDEO_PATH = Path(r"C:\DSP\video.mp4")

assert CHECKPOINT_PATH.exists(), CHECKPOINT_PATH
print("checkpoint:", CHECKPOINT_PATH)
print("image A:", IMAGE_PATH_A)
print("image B:", IMAGE_PATH_B)
print("video  :", VIDEO_PATH)
print("fallback on no-face detection:", USE_IMAGE_FALLBACK_IF_NO_FACE)

checkpoint: C:\DSP\checkpoints\celeba_embedding_best.pt
image A: C:\DSP\Crops\20260309_004904_frame8_face0.jpg
image B: C:\DSP\Crops\20260309_004904_frame151_face0.jpg
video  : C:\DSP\video.mp4
fallback on no-face detection: True


In [29]:
# -----------------------------
# Embedding model
# -----------------------------
class FaceEmbeddingCNN(nn.Module):
    def __init__(self, embedding_dim, num_classes, use_pretrained=False):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if use_pretrained else None
        backbone = models.resnet18(weights=weights)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.embedding = nn.Sequential(
            nn.Linear(in_features, embedding_dim),
            nn.BatchNorm1d(embedding_dim),
            nn.ReLU(inplace=True),
        )
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x):
        features = self.backbone(x)
        embedding = self.embedding(features)
        normalized_embedding = nn.functional.normalize(embedding, p=2, dim=1)
        logits = self.classifier(embedding)
        return normalized_embedding, logits


checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
embedding_dim = checkpoint.get("embedding_dim", 256)
image_size = checkpoint.get("image_size", 224)
num_classes = checkpoint.get("num_classes", 10177)

embedding_model = FaceEmbeddingCNN(embedding_dim=embedding_dim, num_classes=num_classes, use_pretrained=False).to(device)
embedding_model.load_state_dict(checkpoint["model_state_dict"])
embedding_model.eval()

embedding_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

providers = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if device.type == "cuda" else ['CPUExecutionProvider']
face_kwargs = {"name": INSIGHTFACE_MODEL_NAME, "providers": providers}
if INSIGHTFACE_ROOT:
    face_kwargs["root"] = INSIGHTFACE_ROOT
face_app = FaceAnalysis(**face_kwargs)
face_app.prepare(ctx_id=0 if device.type == "cuda" else -1, det_size=DET_SIZE)

print("embedding dim:", embedding_dim)
print("image size   :", image_size)
print("Buffalo loaded")

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\super/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\super/.insightface\models\buffalo_l\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\super/.insightface\models\buffalo_l\det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\super/.insightface\models\buffalo_l\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: C:\Users\super/.insightface\models\buffalo_l\w600k_r50.onnx recognition ['None', 3, 112, 112] 127.

In [30]:
# -----------------------------
# Helpers
# -----------------------------
def pick_largest_face(faces):
    if not faces:
        return None
    return sorted(faces, key=lambda f: (f.bbox[2] - f.bbox[0]) * (f.bbox[3] - f.bbox[1]), reverse=True)[0]


def get_aligned_face(frame_bgr, face_obj, out_size=224):
    if hasattr(face_obj, "kps") and face_obj.kps is not None:
        aligned_bgr = face_align.norm_crop(frame_bgr, landmark=face_obj.kps, image_size=out_size)
        aligned_rgb = cv2.cvtColor(aligned_bgr, cv2.COLOR_BGR2RGB)
        return Image.fromarray(aligned_rgb)

    x1, y1, x2, y2 = face_obj.bbox.astype(int)
    h, w = frame_bgr.shape[:2]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    crop_bgr = frame_bgr[y1:y2, x1:x2]
    if crop_bgr.size == 0:
        return None
    crop_bgr = cv2.resize(crop_bgr, (out_size, out_size))
    crop_rgb = cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB)
    return Image.fromarray(crop_rgb)


def load_image_directly(image_path, out_size=224):
    image = Image.open(image_path).convert("RGB")
    return image.resize((out_size, out_size))


def load_face_from_image(image_path):
    image_path = Path(image_path)
    if not image_path.exists():
        raise FileNotFoundError(image_path)

    frame_bgr = cv2.imread(str(image_path))
    if frame_bgr is None:
        raise RuntimeError(f"Could not read image: {image_path}")

    faces = face_app.get(frame_bgr)
    face = pick_largest_face(faces)
    if face is None:
        if USE_IMAGE_FALLBACK_IF_NO_FACE:
            print(f"No face detected by Buffalo in {image_path.name}; using full image as fallback.")
            return load_image_directly(image_path, out_size=FACE_OUT_SIZE)
        raise RuntimeError(f"No face detected in image: {image_path}")

    aligned = get_aligned_face(frame_bgr, face, out_size=FACE_OUT_SIZE)
    if aligned is None:
        if USE_IMAGE_FALLBACK_IF_NO_FACE:
            print(f"Alignment failed for {image_path.name}; using full image as fallback.")
            return load_image_directly(image_path, out_size=FACE_OUT_SIZE)
        raise RuntimeError(f"Could not align face from image: {image_path}")
    return aligned


@torch.no_grad()
def embed_pil_images(images):
    batch = torch.cat([embedding_transform(image).unsqueeze(0) for image in images], dim=0).to(device)
    embeddings, _ = embedding_model(batch)
    return embeddings.cpu()


def cosine_similarity(a, b):
    a = a / a.norm(p=2)
    b = b / b.norm(p=2)
    return torch.dot(a, b).item()


def pairwise_cosine_matrix(embeddings):
    normed = embeddings / embeddings.norm(p=2, dim=1, keepdim=True)
    return normed @ normed.T


def sample_video_faces(video_path, max_frames=12, seed=42):
    video_path = Path(video_path)
    if not video_path.exists():
        raise FileNotFoundError(video_path)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames <= 0:
        cap.release()
        raise RuntimeError("Could not determine total frame count")

    sample_count = min(max_frames, total_frames)
    rng = random.Random(seed)
    frame_indices = sorted(rng.sample(range(total_frames), sample_count))

    records = []
    progress = tqdm(frame_indices, desc="sample video", dynamic_ncols=True)
    for frame_idx in progress:
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame_bgr = cap.read()
        if not ok:
            continue
        faces = face_app.get(frame_bgr)
        face = pick_largest_face(faces)
        if face is None:
            continue
        aligned = get_aligned_face(frame_bgr, face, out_size=FACE_OUT_SIZE)
        if aligned is None:
            continue
        records.append({
            "frame_index": frame_idx,
            "face": aligned,
        })
        progress.set_postfix({"faces": len(records)})

    cap.release()
    return records


def summarize_video_embeddings(video_embeddings, frame_indices):
    if video_embeddings.size(0) < 2:
        raise RuntimeError("Need at least 2 video face embeddings for consistency analysis")

    matrix = pairwise_cosine_matrix(video_embeddings)
    off_diag = matrix[~torch.eye(matrix.size(0), dtype=torch.bool)]
    mean_embedding = video_embeddings.mean(dim=0)
    mean_embedding = mean_embedding / mean_embedding.norm(p=2)
    to_mean = torch.mv(video_embeddings / video_embeddings.norm(p=2, dim=1, keepdim=True), mean_embedding)

    print("sampled frames:", frame_indices)
    print("num embeddings:", video_embeddings.size(0))
    print("pairwise cosine mean:", f"{off_diag.mean().item():.4f}")
    print("pairwise cosine min :", f"{off_diag.min().item():.4f}")
    print("pairwise cosine max :", f"{off_diag.max().item():.4f}")
    print("similarity to mean  :", [round(x, 4) for x in to_mean.tolist()])
    print("pairwise matrix:\n", matrix)

    return {
        "pairwise_matrix": matrix,
        "pairwise_mean": off_diag.mean().item(),
        "pairwise_min": off_diag.min().item(),
        "pairwise_max": off_diag.max().item(),
        "to_mean": to_mean,
        "mean_embedding": mean_embedding,
    }

In [31]:
# -----------------------------
# Image vs image comparison
# -----------------------------
if IMAGE_PATH_A.exists() and IMAGE_PATH_B.exists():
    face_a = load_face_from_image(IMAGE_PATH_A)
    face_b = load_face_from_image(IMAGE_PATH_B)
    emb_a, emb_b = embed_pil_images([face_a, face_b])
    sim_ab = cosine_similarity(emb_a, emb_b)

    print("image A:", IMAGE_PATH_A)
    print("image B:", IMAGE_PATH_B)
    print("embedding similarity:", f"{sim_ab:.4f}")
else:
    print("Set IMAGE_PATH_A and IMAGE_PATH_B to valid files before running this cell.")

No face detected by Buffalo in 20260309_004904_frame8_face0.jpg; using full image as fallback.
No face detected by Buffalo in 20260309_004904_frame151_face0.jpg; using full image as fallback.
image A: C:\DSP\Crops\20260309_004904_frame8_face0.jpg
image B: C:\DSP\Crops\20260309_004904_frame151_face0.jpg
embedding similarity: 0.6610


In [ ]:
# -----------------------------
# Video consistency check
# -----------------------------
video_consistency = None
if VIDEO_PATH.exists():
    video_records = sample_video_faces(VIDEO_PATH, max_frames=MAX_VIDEO_FRAMES, seed=RANDOM_SEED)
    if len(video_records) < 2:
        raise RuntimeError(f"Need at least 2 detected faces from the video, got {len(video_records)}")

    video_embeddings = embed_pil_images([record["face"] for record in video_records])
    video_consistency = summarize_video_embeddings(
        video_embeddings,
        [record["frame_index"] for record in video_records],
    )
else:
    print("Set VIDEO_PATH to a valid file before running this cell.")

In [ ]:
# -----------------------------
# Video vs image comparison
# -----------------------------
if VIDEO_PATH.exists() and IMAGE_PATH_A.exists():
    if video_consistency is None:
        video_records = sample_video_faces(VIDEO_PATH, max_frames=MAX_VIDEO_FRAMES, seed=RANDOM_SEED)
        if len(video_records) < 1:
            raise RuntimeError("No usable faces found in video")
        video_embeddings = embed_pil_images([record["face"] for record in video_records])
        mean_embedding = video_embeddings.mean(dim=0)
        mean_embedding = mean_embedding / mean_embedding.norm(p=2)
    else:
        mean_embedding = video_consistency["mean_embedding"]

    face_a = load_face_from_image(IMAGE_PATH_A)
    image_embedding = embed_pil_images([face_a])[0]
    sim_video_to_image = cosine_similarity(mean_embedding, image_embedding)

    print("video:", VIDEO_PATH)
    print("image:", IMAGE_PATH_A)
    print("video mean vs image similarity:", f"{sim_video_to_image:.4f}")
else:
    print("Set VIDEO_PATH and IMAGE_PATH_A to valid files before running this cell.")